In [11]:
import pdfplumber
import re

pages_data = []

with pdfplumber.open("product_specs_amazon_style.pdf") as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        text = clean_text(text)
        tables = page.extract_tables()

        products = re.split(r'\n(?=[A-Z][A-Za-z0-9\s"]+\n)', text)

        pages_data.append({
            "text": text,
            "tables": tables,
            "products": products
        })

In [12]:
pages_data

[{'text': 'Smart LED TV 55" Brand: Samsung Price: 599.99 USD About this item: 4K Ultra HD resolution for crystal-clear viewing Smart TV with built-in Wi-Fi and streaming apps HDR support for vivid colors and contrast Multiple HDMI and USB ports for connectivity Technical Details: Specification Details Display Size 55 inches Resolution 3840 x 2160 (4K UHD) Connectivity Wi-Fi, HDMI, USB Audio Dolby Digital Surround Dimensions 48.7 x 28.1 x 2.3 inches Weight 15.5 kg Warranty 1 Year Manufacturer Warranty What\'s in the Box Smart LED TV 55", User Manual, Warranty Card Wireless Bluetooth Headphones Brand: Sony Price: 199.99 USD About this item: Noise-cancelling over-ear design Up to 30 hours of battery life Bluetooth 5.0 wireless connectivity Comfort-fit cushioned earcups Technical Details: Specification Details Type Over-Ear Connectivity Bluetooth 5.0 Battery Life 30 hours Charging Time 2 hours',
  'tables': [[['Specification', 'Details'],
    ['Display Size', '55 inches'],
    ['Resolution

In [20]:
def split_products(text: str):
    """
    Split text into product blocks using Brand as anchor
    """
    blocks = re.split(r'(?=\nBrand:)', text)

    return [b.strip() for b in blocks if b.strip()]

In [1]:
def clean_text(text: str) -> str:
    if not text:
        return ""

    text = re.sub(r'\(cid:\d+\)', '•', text)

    text = re.sub(r'\s+', ' ', text)

    text = text.replace(" About this item:", "\nAbout this item:")
    text = text.replace(" Technical Details:", "\nTechnical Details:")
    text = text.replace(" Brand:", "\nBrand:")
    text = text.replace(" Price:", "\nPrice:")

    return text.strip()

In [23]:
def extract_basic_info(block: str):
    name = block.split("\n")[0].strip()

    brand_match = re.search(r'Brand:\s*(.*)', block)
    price_match = re.search(r'Price:\s*([\d.]+)', block)

    return {
        "product_name": name,
        "brand": brand_match.group(1).strip() if brand_match else None,
        "price": float(price_match.group(1)) if price_match else None,
    }

In [24]:
def extract_features(block: str):
    features = []

    if "About this item:" in block:
        section = block.split("About this item:")[1]

        # stop at next section
        section = section.split("Technical Details:")[0]

        lines = section.split("•")

        for line in lines:
            line = line.strip()
            if line:
                features.append(line)

    return features

In [25]:
def extract_specs(table):
    specs = {}

    if not table:
        return specs

    for row in table:
        if len(row) == 2:
            key = row[0].strip() if row[0] else None
            value = row[1].strip() if row[1] else None

            if key and value:
                specs[key] = value

    return specs

In [26]:
def parse_pdf(file_path: str):
    results = []

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            raw_text = page.extract_text()
            tables = page.extract_tables()

            cleaned = clean_text(raw_text)
            product_blocks = split_products(cleaned)

            for i, block in enumerate(product_blocks):
                basic = extract_basic_info(block)
                features = extract_features(block)

                specs = {}
                if i < len(tables):
                    specs = extract_specs(tables[i])

                product = {
                    **basic,
                    "features": features,
                    "specifications": specs
                }

                results.append(product)
    return results

In [28]:
def merge_products(raw_products):
    merged = []
    current = None

    for item in raw_products:
        has_brand = item.get("brand") is not None
        has_price = item.get("price") is not None

        # 👉 new product starts
        if has_brand and has_price:
            if current:
                merged.append(current)

            current = item

        else:
            # 👉 merge into previous
            if not current:
                continue

            # merge name
            if current["product_name"].startswith("Brand:"):
                current["product_name"] = item.get("product_name")

            # merge features
            current["features"].extend(item.get("features", []))

            # merge specs
            current["specifications"].update(item.get("specifications", {}))

    if current:
        merged.append(current)

    return merged

In [29]:
def post_process(products):
    for p in products:
        # fix product name
        if p["product_name"].startswith("Brand:"):
            p["product_name"] = None

        # remove junk key
        p["specifications"].pop("Specification", None)

    return products

In [31]:
raw  = parse_pdf("product_specs_amazon_style.pdf")
merged = merge_products(raw)
final = post_process(merged)

In [32]:
final

[{'product_name': None,
  'brand': 'Samsung',
  'price': 599.99,
  'features': ['4K Ultra HD resolution for crystal-clear viewing',
   'Smart TV with built-in Wi-Fi and streaming apps',
   'HDR support for vivid colors and contrast',
   'Multiple HDMI and USB ports for connectivity'],
  'specifications': {'Type': 'Over-Ear',
   'Connectivity': 'Bluetooth 5.0',
   'Battery Life': '30 hours',
   'Charging Time': '2 hours'}},
 {'product_name': "Weight 250 g Warranty 1 Year Manufacturer Warranty What's in the Box Wireless Bluetooth Headphones, User Manual, Warranty Card Smartphone X15",
  'brand': 'Sony',
  'price': 199.99,
  'features': ['Noise-cancelling over-ear design',
   'Up to 30 hours of battery life',
   'Bluetooth 5.0 wireless connectivity',
   'Comfort-fit cushioned earcups'],
  'specifications': {'Weight': '250 g',
   'Warranty': '1 Year Manufacturer Warranty',
   "What's in the Box": 'Wireless Bluetooth Headphones, User Manual, Warranty Card'}},
 {'product_name': None,
  'bran